# Validation against an independent calculator

To check the model's correctness, this notebook compares it against a
third-party reference: the **"UK Buy vs Rent Calculator" spreadsheet by
Financial Interest** (2025 stamp duty rules). The comparison is done in three
steps:

1. **Building blocks** — stamp duty, mortgage payment, and amortisation
   balances should match exactly, since both models implement the same rules.
2. **Headline result** — run both models on identical assumptions.
3. **Reconciliation** — explain any gap by replicating the reference
   calculator's methodology formula-by-formula.

**Spoiler:** the building blocks match to the penny; the headline results
differ by ~£23k; and the gap traces to an off-by-one in the *reference*
calculator, which charges 7 years of costs against only 6 years of equity
growth. Correcting that single inconsistency reconciles the two models to
within £550 (≈0.7%).

In [1]:
from rent_vs_buy import SimulationConfig, run_simulation, stamp_duty, monthly_payment
from rent_vs_buy.mortgage import balance_after

## 1. The reference calculator's assumptions

Taken directly from its input sheet: £500k property, £50k deposit (10%),
4.5% mortgage over 30 years, first-time buyer, 2.5% house appreciation,
£5k buying costs + £5k initial repairs, 1% selling fee, 1% annual maintenance,
£500/yr insurance, £2,200/month rent growing 3%/yr, savings returning 2.5%,
and a 7-year horizon. Its verdict: **"After 7 year(s), Buying is better by
£59,034"** (buyer net −£129,839, renter net −£188,873).

In [2]:
cfg = SimulationConfig(
    house_price=500_000, deposit_fraction=0.10, house_growth_rate=0.025,
    first_time_buyer=True,
    mortgage_rate=0.045, mortgage_term_years=30,
    purchase_legal_costs=10_000,          # their £5k buying costs + £5k repairs
    maintenance_rate=0.01, insurance_annual=500,
    selling_cost_rate=0.01,
    monthly_rent=2_200, rent_growth_rate=0.03,
    equity_return_rate=0.025, fund_fee_rate=0.0, invest_in_isa=True,
    years=7,
)

# Values recorded from the reference spreadsheet
SHEET = {
    "stamp_duty": 10_000.0,
    "annual_mortgage_payment": 27_361.00673,
    "balance_after_6y": 401_123.1903,
    "balance_after_7y": 391_618.2781,
    "buyer_net": -129_839.1461,
    "renter_net": -188_872.6813,
    "difference": 59_033.53515,
}

## 2. Building blocks — should match exactly

In [3]:
checks = {
    "Stamp duty (FTB, £500k)": (stamp_duty(500_000, True), SHEET["stamp_duty"]),
    "Monthly mortgage payment": (monthly_payment(450_000, 0.045, 30),
                                 SHEET["annual_mortgage_payment"] / 12),
    "Mortgage balance after 6y": (balance_after(450_000, 0.045, 30, 72),
                                  SHEET["balance_after_6y"]),
    "Mortgage balance after 7y": (balance_after(450_000, 0.045, 30, 84),
                                  SHEET["balance_after_7y"]),
}
print(f"{'check':<28}{'this model':>15}{'reference':>15}{'diff':>10}")
for name, (ours, theirs) in checks.items():
    print(f"{name:<28}{ours:>15,.2f}{theirs:>15,.2f}{ours - theirs:>10.2f}")

check                            this model      reference      diff
Stamp duty (FTB, £500k)           10,000.00      10,000.00      0.00
Monthly mortgage payment           2,280.08       2,280.08      0.00
Mortgage balance after 6y        401,123.19     401,123.19     -0.00
Mortgage balance after 7y        391,618.28     391,618.28     -0.00


## 3. Headline result on identical assumptions

In [4]:
result = run_simulation(cfg)
print(f"This model:  buying better by £{result.final_advantage:,.0f}")
print(f"Reference:   buying better by £{SHEET['difference']:,.0f}")
print(f"Gap:         £{result.final_advantage - SHEET['difference']:,.0f}")

This model:  buying better by £82,240
Reference:   buying better by £59,034
Gap:         £23,207


## 4. Reconciling the gap

Replicate the reference calculator's methodology exactly, from its formulas.
Two features of its design matter:

- **Indexing.** Its year-*k* table row holds the mortgage balance and house
  value after *k − 1* years, but its cost summation over "7 years" includes
  rows 1–7 — i.e. **7 years of payments, maintenance, insurance and rent are
  charged against only 6 years of appreciation and amortisation.**
- **Renter investing.** Its renter invests only `max(mortgage payment − rent, 0)`
  — the maintenance/insurance cost difference is not invested (in this model
  the renter invests the full housing-cost difference, keeping the two
  budgets symmetric).

In [5]:
price, rate = 500_000, 0.045
pay_m = monthly_payment(450_000, rate, 30)
hg, rent_g, save_r = 0.025, 0.03, 0.025

# --- Buyer, their way: 7 years of costs, 6 years of equity -------------------
payments_7y = 7 * pay_m * 12
maint_7y = sum(price * (1 + hg) ** k * 0.01 for k in range(7))
value_6y = price * (1 + hg) ** 6                       # the off-by-one
equity_6y = value_6y - balance_after(450_000, rate, 30, 72)
buyer_net = (-50_000 - 10_000 - payments_7y + equity_6y
             - 5_000 - 5_000 - maint_7y - 500 * 7 - value_6y * 0.01)

# --- Renter, their way -------------------------------------------------------
pot0 = 50_000 + 10_000 + 5_000 + 5_000 + 500           # incl. its £500 insurance quirk
ret_initial = pot0 * ((1 + save_r) ** 7 - 1)
rents = [2_200 * (1 + rent_g) ** k for k in range(7)]
rent_paid = sum(r * 12 for r in rents)
Q = [max(pay_m - r, 0) * 12 for r in rents]
R = 0.0
for q in Q:
    R = (R + q) * (1 + save_r)
renter_net = ret_initial + (R - sum(Q)) - rent_paid

print(f"Replicated buyer net:   £{buyer_net:>12,.0f}   (sheet: £{SHEET['buyer_net']:,.0f})")
print(f"Replicated renter net:  £{renter_net:>12,.0f}   (sheet: £{SHEET['renter_net']:,.0f})")
print(f"Replicated difference:  £{buyer_net - renter_net:>12,.0f}   (sheet: £{SHEET['difference']:,.0f})")

Replicated buyer net:   £    -129,839   (sheet: £-129,839)
Replicated renter net:  £    -188,778   (sheet: £-188,873)
Replicated difference:  £      58,939   (sheet: £59,034)


The replication lands within ~£100 of the sheet's own numbers, confirming
the methodology was captured correctly. Now apply **one change only**: credit
the buyer with the full 7 years of appreciation and amortisation, matching the
7 years of costs already charged.

In [6]:
value_7y = price * (1 + hg) ** 7
equity_7y = value_7y - balance_after(450_000, rate, 30, 84)
buyer_net_fixed = (-50_000 - 10_000 - payments_7y + equity_7y
                   - 5_000 - 5_000 - maint_7y - 500 * 7 - value_7y * 0.01)

print(f"Reference method, off-by-one corrected: £{buyer_net_fixed - renter_net:,.0f}")
print(f"This model:                              £{result.final_advantage:,.0f}")

Reference method, off-by-one corrected: £82,795
This model:                              £82,240


## Conclusion

| comparison | buying advantage at 7 years |
|---|---|
| Reference calculator, as published | £59,034 |
| Reference method, equity off-by-one corrected | ~£82,800 |
| This model | ~£82,240 |

- All shared financial primitives — stamp duty bands, annuity payments,
  amortisation — **agree to the penny**.
- The published £23k gap is explained by an indexing inconsistency in the
  reference calculator: 7 years of costs are netted against a 6-year equity
  position.
- After correcting it, the residual ~£550 (≈0.7%) comes from deliberate
  design differences: this model compounds monthly rather than annually,
  invests the *full* housing-cost difference for the renter, and excludes
  the reference's £500 insurance line from the renter's initial pot.

Validating against an independent implementation is exactly how a small
indexing bug like this gets caught — in either direction.